# Multimodal Graph Neural Networks for Movie Recommendation

## Architecture Overview
This project implements a **Hybrid Multimodal Recommendation System** integrating Computer Vision (CV), Natural Language Processing (NLP), and Graph Neural Networks (GNNs).

The goal is to solve the item-cold-start and sparse-interaction challenges in collaborative filtering by enriching items with multimodal representations:
1. **Computer Vision (CV):** Multi-label genre classification on TMDB movie posters using a custom CNN architecture benchmarked against a fine-tuned ResNet50 model. Visual representations are also extracted.
2. **Natural Language Processing (NLP):** Multi-label genre classification from textual plot summaries using sequence modeling ( Transformer-based embeddings).
3. **Graph Neural Networks (GNN):** Bipartite user-movie interaction graph (`ml-latest-small`) modeled via PyTorch Geometric. Nodes are initialized with  the   Visual and Textual embeddings that were created, and a Relational Graph Convolutional Network (RGCN / Hetero-GraphSAGE) predicts user-item links.

## 1. Data Pipeline & Stratified Sourcing

To train robust feature extractors without target data leakage:
- **Graph Target (`ml-latest-small`):** It is a bipartite graph consisting of almost 600 users and 9,000 movies.
- **Multimodal Training Pool (`ml-25m`):**    This is a massive dataset. We will extract a disjoint subset of movies from this to train the CV and NLP models, ensuring our models learn on separate data.

- **Stratified Distribution Matching:** Stratified sampling matching `Decade + Primary Genre` distributions between the pre-training set and the evaluation graph ensures domain alignment.


## Imports


In [ ]:
# Standard Library
import asyncio
from collections import Counter
from getpass import getpass
import os
import re

#Data Handling
import aiohttp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import seaborn as sns
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import MultiLabelBinarizer
from tqdm.auto import tqdm

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.models as models
from torchvision import transforms

# PyTorch Geometric (GNNs)
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv
import torch_geometric.transforms as T

In [ ]:

# Create a datasets directory
os.makedirs('datasets', exist_ok=True)

# Download and unzip the datasets
!wget -q https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!wget -q https://files.grouplens.org/datasets/movielens/ml-25m.zip
!unzip -q -o ml-latest-small.zip -d datasets/
!unzip -q -o ml-25m.zip -d datasets/

---
### **Stratified Sampling**
To ensure our CV and NLP models generalize well to the final ml-latest-small graph, stratified sampling is performed. We calculate the distribution of 'Decade + Primary Genre' in the small dataset, and force our training samples to match that exact distribution. The main idea is that the training sample must not be purely random. A random subset of 15,000 movies from the massive 25M dataset would definitely lead to less frequent movie genres being underrepresented, eventually resulting in a distribution shift.

In [ ]:
def extract_metadata(df):
    """Extracts Decade and Primary Genre to create a sampling stratum."""
    df['year'] = df['title'].str.extract(r'\((\d{4})\)').astype(float)
    df['decade'] = (df['year'] // 10 * 10).fillna(0).astype(int)
    df['primary_genre'] = df['genres'].str.split('|').str[0]
    df['stratum'] = df['decade'].astype(str) + "_" + df['primary_genre']
    return df

In [ ]:
print("Loading datasets...")
small_movies = pd.read_csv('datasets/ml-latest-small/movies.csv')
small_links = pd.read_csv('datasets/ml-latest-small/links.csv')

large_movies = pd.read_csv('datasets/ml-25m/movies.csv')
large_links = pd.read_csv('datasets/ml-25m/links.csv')

print("Processing metadata for stratified sampling...")
small_movies = extract_metadata(small_movies)
large_movies = extract_metadata(large_movies)

# Calculate target distribution from the small graph
target_distribution = small_movies['stratum'].value_counts(normalize=True)


We prevent data leakage by creating a **disjoint** set of 15,000 movies to use for training on the image/text modalities. This is vital for the model's generalization ability!

In [ ]:
# Find disjoint set (Movies in 25M that are NOT in latest-small)
small_ids = set(small_movies['movieId'])
disjoint_movies = large_movies[~large_movies['movieId'].isin(small_ids)].copy()

# Merge with TMDB links
# Discard movies with missing TMDB ids (database for plot summaries/posters)
disjoint_full = pd.merge(disjoint_movies, large_links, on='movieId')
disjoint_full = disjoint_full.dropna(subset=['tmdbId'])
disjoint_full['tmdbId'] = disjoint_full['tmdbId'].astype(int)

# Map weights and sample
disjoint_full['sample_weight'] = disjoint_full['stratum'].map(target_distribution).fillna(0)
train_subset = disjoint_full.sample(n=15000, weights='sample_weight', random_state=42)

# Merge small_movies with their TMDB links
small_full = pd.merge(small_movies, small_links, on='movieId')
small_full = small_full.dropna(subset=['tmdbId'])
small_full['tmdbId'] = small_full['tmdbId'].astype(int)

columns_to_keep = ['movieId', 'title', 'genres', 'tmdbId']
train_subset = train_subset[columns_to_keep]
small_full = small_full[columns_to_keep]

In [ ]:
print(f"Stratified Training Subset: {len(train_subset)} movies.")
print(f"ML-Small Inference Target: {len(small_full)} movies.")

## **Multi-Label Genre Classification of Movie Posters**

Firstly comes the training of a convolutional model to predict movie genres based on the poster images and extracting the produced visual features (visual embeddings)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Assuming datasets.zip is in the root of your Google Drive
!unzip -o /content/drive/MyDrive/datasets.zip -d datasets/

In [ ]:
# Load train_metadata_full.csv into train_subset
train_subset = pd.read_csv('./datasets/datasets/train_metadata_full.csv')
small_full = pd.read_csv('./datasets/datasets/ml_small_metadata_full.csv')

print("train_metadata_full.csv loaded successfully into train_subset.")
print("ml_small_metadata_full.csv loaded successfully into small_full.")

**Multi-Hot Encoding the Target Variables**

You will frame genre prediction as a **Multi-Label Classification** problem. The required loss function expects the target to be a binary matrix (0s and 1s).

However, the genres look like this:

In [ ]:
train_subset['genres'].head()

In [ ]:
# binarizer for multi-label classification
mlb = MultiLabelBinarizer()

#encode_genres( genres as multi-hot binary indicator columns)
def encode_genres(df):

    # Preprocess the genres as a list which is the correct format for the binarizer.
    genre_lists = df['genres'].fillna('').apply(lambda x: x.split('|') if x != '' else [])
    # Fit/transform the genre lists into a binary multi-hot matrix
    genre_matrix = mlb.fit_transform(genre_lists)
    genre_df = pd.DataFrame(genre_matrix, columns=mlb.classes_, index=df.index)
    df = pd.concat([df, genre_df], axis=1)

    #final data frame for the movies.
    return df

train_subset = encode_genres(train_subset)
small_full = encode_genres(small_full)

print(f"Total unique genres: {len(mlb.classes_)}")
display(train_subset.head(3))

The size of the  TMDB posters.

In [ ]:
id = train_subset.iloc[42]['tmdbId']

# Define the image directory
IMAGE_DIR = "./datasets/datasets/posters/"

# Construct the image path
image_filename = f"{id}.jpg"
image_path = os.path.join(IMAGE_DIR, image_filename)


image = Image.open(image_path)

# Display the image
plt.imshow(image)
plt.title(f"Original Movie Poster")
plt.axis('off')
plt.show()

# Print its dimensions
print(f"Image dimensions: {image.size[1]}x{image.size[0]} pixels (Height x Width)")

**Exploratory Data Analysis**



In [ ]:
genre_columns = mlb.classes_

# occurrences of each genre
genre_counts = train_subset[genre_columns].sum().sort_values(ascending=False)

# Plot of the distribution
plt.figure(figsize=(12, 6))
sns.barplot(x=genre_counts.values, y=genre_counts.index, hue=genre_counts.index, palette="viridis", legend=False)
plt.title("Genre Distribution in Training Dataset (Label Imbalance)", fontsize=14)
plt.xlabel("Number of Movies", fontsize=12)
plt.ylabel("Genre", fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

#  top and bottom 3
print("Top 3 Most Frequent Genres:")
print(genre_counts.head(3))
print("\nBottom 3 Least Frequent Genres:")
print(genre_counts.tail(3))

We will crop the posters in order to be easier when it comes to the design of the CNN. A resolution of **128x128 pixels** will be used. The reshaped posters will be packed into a ``MoviePosterDataset`` class.

In [ ]:
class MoviePosterDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

        # Get the label columns directly from mlb.classes_
        # This ensures we always select only the genre columns by their names
        self_label_cols_ = list(mlb.classes_)
        self.label_cols = [col for col in self.dataframe.columns if col in self_label_cols_]

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = self.dataframe.loc[idx, 'poster_file']
        img_path = os.path.join(self.image_dir, img_name)

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        labels = torch.tensor(self.dataframe.loc[idx, self.label_cols].astype(np.float32).values, dtype=torch.float32)


        return image, labels


target_size = 128

# Define transforms (Resize, CenterCrop, ToTensor, Normalize)
poster_transform = transforms.Compose([
    transforms.Resize(target_size),
    transforms.CenterCrop((target_size, target_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

poster_aug_transform = transforms.Compose([
    transforms.Resize(target_size),
    transforms.RandomHorizontalFlip(p=0.4),
    transforms.RandomRotation(degrees=5),
    transforms.CenterCrop((target_size, target_size)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [ ]:
from sklearn.model_selection import train_test_split

#Train and Validation (80/20)
train_df, val_df = train_test_split(train_subset, test_size=0.2, random_state=42)

print("Initializing Datasets...")
train_dataset = MoviePosterDataset(dataframe=train_df, image_dir=IMAGE_DIR, transform=poster_aug_transform)
val_dataset = MoviePosterDataset(dataframe=val_df, image_dir=IMAGE_DIR, transform=poster_transform)

# ML-Small dataset as the Test Set
test_dataset = MoviePosterDataset(dataframe=small_full, image_dir=IMAGE_DIR, transform=poster_transform)

In [ ]:
img, label = train_dataset[0]

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for ax in axes.flatten():
    aug_img, _ = train_dataset[0]   # same index with new random augment each time

    aug_img = aug_img.numpy().transpose(1, 2, 0)
    aug_img = std * aug_img + mean
    aug_img = np.clip(aug_img, 0, 1)

    ax.imshow(aug_img)
    ax.axis("off")

plt.tight_layout()
plt.show()

Saving the `train_df, val_df` using the pickle format.




In [ ]:
import pickle as pkl
pkl.dump(train_df, open("train_df.pkl", "wb"))
pkl.dump(val_df, open("val_df.pkl", "wb"))

The "sum up"

In [ ]:
image, labels = train_dataset[42]
# Convert tensor to numpy array and move channels to the last dimension (H, W, C)
image = image.numpy().transpose((1, 2, 0))
# Clip values to [0, 1] as some values might be slightly outside due to floating point operations
image = np.clip(image, 0, 1)

# Get the genre names corresponding to the '1's in the labels
label_cols = train_dataset.label_cols
active_genres = [genre for i, genre in enumerate(label_cols) if labels[i] == 1]
genres_str = ', '.join(active_genres) if active_genres else 'No genres listed'

# Display
plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.title(f"Processed Movie Poster\nGenres: {genres_str}")
plt.axis('off')
plt.show()

In [ ]:
batch_size = 32

# train, validation and test loaders

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train Size: {len(train_dataset)} | Val Size: {len(val_dataset)} | Test (ML-Small) Size: {len(test_dataset)}")

print("For Training: ")
print(f"Number of Batches per Epoch: {len(train_loader)}")

sample_images, sample_labels = next(iter(train_loader))
print(f"Image Batch Shape: {sample_images.shape} -> [Batch, Channels, Height, Width]")
print(f"Label Batch Shape: {sample_labels.shape} -> [Batch, Num_Genres]")

---
### **Model Architecture (Multi-Label Classification)**


As mentioned before  genre predicting is a Multi-Label Classification problem. A movie can be 3 differents genres simultaneously. Therefore a CNN feature extractor using `nn.Conv2d`, `nn.ReLU`, and `nn.MaxPool2d` will be designed. A typical architecture stacks 3 to 4 of these blocks. We will Flatten the spatial dimensions and pass the features through a fully connected (`nn.Linear`) classification head. The final output dimension must exactly match the number of unique genres in our dataset. It is worth mentioning that we will not apply a Softmax or Sigmoid layer at the very end of your `forward` pass, because PyTorch's recommended multi-label loss function expects raw, unnormalized logits.



In [ ]:
#CustomCNN
class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super(CustomCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),                  #128 to 64

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),                  #64 to 32

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),                  #32 to 16

            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),                  #16 to 8

            nn.Flatten()

        )

        # size of the flattened tensor after the feature extractor.

        self.flatten_dim = 256 * 8 * 8

        #Definition of the classification head. Use of nn.Linear layers and nn.Dropout to prevent overfitting
        self.fc1 = nn.Linear(self.flatten_dim, 512)
        self.dropout1 = nn.Dropout(0.3)

        self.fc2 = nn.Linear(512, num_classes)


    def forward(self, x):
        #the forward pass
        x = self.features(x)
        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = self.fc2(x)
        return x

    def extract_embeddings(self, x):
        x = self.features(x)
        x = F.relu(self.fc1(x))
        return x

In [ ]:
# Initialize the model
num_genres = len(mlb.classes_)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cv_model = CustomCNN(num_classes=num_genres).to(device)
print(f"Model initialized on: {device}")

---
### **Training  & Loss Function selection**
To train this model, we need a loss function capable of handling multiple correct labels per image.A standard `CrossEntropyLoss`  will not work here, as it assumes only one class can be true.

Our dataset is heavily imbalanced. If we train normally, the model will just learn to predict the most common genres  for every poster.To fix this problem, we will calculate the Positive Weigh for each genre. This tells the loss function to assign a higher penalty when the model makes a mistake on a rare genre.


In [ ]:
# number of positive occurrences for each genre in the training set
pos_counts = train_df[genre_columns].sum().values
total_samples = len(train_df)

# pos_weight: (negatives / positives)
pos_weights = (total_samples - pos_counts) / pos_counts
#  damp extreme values
smoothed_weights = np.sqrt(pos_weights)

# Convert to PyTorch tensor
pos_weight_tensor = torch.tensor(smoothed_weights, dtype=torch.float32).to(device)

print("Calculated pos_weight for the first 5 genres:")
for i in range(5):
    print(f"{genre_columns[i]}: {pos_weight_tensor[i].item():.2f}")

In [ ]:
criterion = nn.BCEWithLogitsLoss(pos_weight = pos_weight_tensor) #multi-label classification.
optimizer = torch.optim.AdamW(cv_model.parameters(), lr=1e-3, weight_decay=1e-4)


epochs = 50

train_losses = []
val_losses = []

for epoch in range(epochs):
    # TRAINING
    cv_model.train()
    running_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")

    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = cv_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()


        # Loss update
        running_loss += loss.item() * images.size(0)
        progress_bar.set_postfix({'batch_loss': loss.item()})


    train_loss = running_loss / len(train_dataset)
    train_losses.append(train_loss) # Store for plotting

    # VALIDATION
    cv_model.eval()
    val_loss = 0.0

    # torch.no_grad() prevents PyTorch from storing gradients, saving memory
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            # model's predictions and  the batch loss

            outputs = cv_model(images)
            loss = criterion(outputs, labels)

            # Loss update
            val_loss += loss.item() * images.size(0)

    val_loss = val_loss / len(val_dataset)
    val_losses.append(val_loss) # Store for plotting

    print(f"End of Epoch {epoch+1} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

print("Training complete!")

In [ ]:
# VISUALIZE LOSS CURVES
plt.figure(figsize=(10, 5))
plt.plot(range(1, epochs+1), train_losses, label='Training Loss', marker='o', color='blue')
plt.plot(range(1, epochs+1), val_losses, label='Validation Loss', marker='o', color='orange')
plt.title('Training and Validation Loss Over Epochs', fontsize=14)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.xticks(range(1, epochs+1))
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
torch.save(cv_model.state_dict(), "/content/drive/MyDrive/cv_model_best3.pth")

---
### **Model Evaluation & Qualitative Analysis**

Now that your Custom CNN is trained, we need to evaluate how well it actually learned to identify genres. In multi-label classification the mpore tradiotional metrics do not work that well and could be strict and misleading. For example, if a movie is truly "Action|Sci-Fi|Thriller" and your model predicts "Action|Sci-Fi", it is very close to being completely correct, however the "accuracy" would be 0.To properly evaluate your model, we will  calculate metrics like F1-Score and ROC-AUC




In order to correctly ebvaluate the model we will pass your model's raw output logits through a `torch.sigmoid()` function to convert them into probabilities (0.0 to 1.0). Then we will Convert those probabilities into binary predictions (0 or 1) using a threshold, and after that we will Calculate the Macro and Micro F1-Score. The Macro calculates the F1-score for each genre independently and then takes the unweighted average. On the other hand the Micro calculates the F1-score globally by counting the total true positives, false negatives, and false positives across all classes.

In [ ]:
# Quantitative Evaluation
cv_model.eval()


all_targets = []
all_predictions = []

print("Evaluating model on the Test Set (ML-Small)...")


# EVALUATION LOOP
with torch.no_grad():
    for images, labels in test_loader:
      images = images.to(device)
      labels = labels.to(device)

      logits = cv_model(images)
      probs = torch.sigmoid(logits)

      all_targets.append(labels.cpu().numpy())
      all_predictions.append(probs.cpu().numpy())

all_targets = np.vstack(all_targets)
all_predictions = np.vstack(all_predictions)

binary_predictions = (all_predictions >= 0.5).astype(int)

#metrics
macro_f1 = f1_score(all_targets, binary_predictions, average='macro', zero_division=0)
micro_f1 = f1_score(all_targets, binary_predictions, average="micro", zero_division=0)
roc_auc = roc_auc_score(all_targets, all_predictions, average = 'macro')


#VISUALIZE  METRICS
metrics_dict = {'Macro F1': macro_f1, 'Micro F1': micro_f1, 'ROC-AUC': roc_auc}
plt.figure(figsize=(8, 5))
bars = plt.bar(metrics_dict.keys(), metrics_dict.values(), color=['#87CEEB', '#98FB98', '#FA8072'])
plt.ylim(0, 1.1)
plt.title('Test Set Evaluation Metrics', fontsize=14)
plt.ylabel('Score', fontsize=12)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.02, f'{yval:.4f}',
             ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
pos_weight_tensor



Now we will Visualize the model's predictions to see if they make logical sense.W e will  display 5 random movie posters from the dataset.

In [ ]:
# Qualitative  (Visual Grid)
def imshow_denormalized(img):
    img = img / 2 + 0.5
    img = np.clip(img, 0, 1)
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))

# We Extract exactly ONE batch of images and labels from the test_loader set
images, labels = next(iter(test_loader))

# we run the images through the model to get binary predictions
cv_model.eval()
with torch.no_grad():
    outputs = cv_model(images.to(device))
    probs = torch.sigmoid(outputs)
    preds = (probs > 0.5).int().cpu()
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
genre_names = mlb.classes_
for i in range(5):
    ax = axes[i]
    # For the i-th image in the batch, we extract the list of actual text strings for both the true genres and the predicted ones.
    true_genres = [genre_names[j] for j in range(len(genre_names)) if labels[i][j] == 1]
    pred_genres = [genre_names[j] for j in range(len(genre_names)) if preds[i][j] == 1]

    #VISUALIZATION
    plt.sca(ax)
    imshow_denormalized(images[i])
    ax.axis('off')
    title_text = f"True: {', '.join(true_genres)}\nPred: {', '.join(pred_genres) if pred_genres else 'None'}"
    color = 'green' if set(true_genres) == set(pred_genres) else 'black'
    ax.set_title(title_text, fontsize=10, wrap=True, color=color)
plt.tight_layout()
plt.show()

Common questions:

1. Why is standard "exact-match accuracy" a poor metric? What happens to the F1-Score if *every* movie is predicted as "Drama" and "Comedy"?
2.  What happens if we lower the probability threshold from 0.5 to 0.3? How are the Precision and Recall metrics expected to change?.
3. Which genres did the model perform best on, and which did it struggle with? Does this correlate with the frequency of those genres in the training data?
4. Based on the visual "eye test" of the posters, what is the one genre that is visually easy for a CNN to learn, and what is the one genre that is visually ambiguous?

Answers:
1.

When it comes to a task such as predicting movie genres, which is basically a multi-label classification task, Exact-Match Accuracy is probably not the ideal metric. It is not obligatory for a single movie to be associated with just one genre, on the contrast, it is more common for it to be associated with a combination of them. The Exact-Match Accuracy metric only awards a point if the model identifies the entire set perfectly. If a model predicts, let's say, two out of the three genres correctly, the metric translates the result as a complete failure (0%), even though it is partially correct at 66.6%. Therefore, it fails to provide a more objective view of the model's learning progress and is obviously not a suitable metric!

Furthermore, if a model defaults to predicting the most frequent genres, such as Drama and Comedy, the Micro F1-Score might remain high because these classes dominate the dataset. However, this score is completely deceptive, since the model seems to "ignore" the other 18 genres, proving it has not truly learned to distinguish between different cinematic styles, and the loss would be very large. Finally, this strategy would cause the Macro F1-Score to collapse toward zero. This score takes into consideration each genre separately, and if we predict every movie to be a "Drama" and "Comedy" while a specific movie is neither of them, then ultimately the Macro F1-Score becomes 0.


2.

If the probability threshold is lowered from 0.5 (the typical) to 0.3, we expect a significant shift in the model's behavior. Recall will propably increase. The less "strict" the model becomes  the more likely it is for the model to assign more labels, and therefore  capture a larger portion of the correct genres of a poster. On the contrast , Precision will decrease. By lowering the bar for prediction, the model is more prone to "guessing" genres , leading to an increase in 'False Positives' that are predicted but do not actually apply to the movie. This trade-off is crucial for optimizing the F1-Score, especially for rare genres that rarely achieve high probability scores.


3.

Model performance is generally heavily infuenced by the distribution of the training data. This specific  model seems to have  performed best on  genres like Drama and Comedy (high-frequency genres) because the CNN has been exposed to thousands of examples, allowing it to identify consistent visual patterns associated with them more easily. On the other hand, the model struggles with "long-tail" genres such as Film-Noir, Western, or Musical. This is completely logical if we take into consideration Class Imbalance. A genre that has very few samples, can make it very hard for the model to identify and  generalize its visual features and instantly become a rare choice for it. Therefore the model  often fails to recognize these rare genres, and this is basically why using weighted loss functions or oversampling, are such  crusial steps for the CNN to perform well enough.

4.

Based on the posters, some genres are visually easy for a CNN to learn, while others are very ambiguous. For example, Animation is an easy genre because these posters usually have very bright colors and digital drawings that have very little similarities to  the like real-life photos. In contrast, genres like Thriller and Horror are visually ambiguous. Both types of movies often use dark colors, shadows, and similar fonts to create a feeling of mystery or fear. Their posters are designed with a similar style, and therefore it is very difficult for the model to find unique features to separate one from the other.

### **Optimization: Transfer Learning with Pre-trained ResNet**
Now instead of training a CNN from scratch, we will now leverage Transfer Learning. We will use  a pre-trained ResNet50 and modify its final classification layer to output the correct number of genres, fine-tune it on the training set, and perform the same quantitative evaluation (Macro F1, Micro F1, ROC-AUC) on the ML-Small test set. After that we will compare the CNN and the ResNet50 models.

In [ ]:
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
mlb = MultiLabelBinarizer()

train_subset = encode_genres(train_subset)
small_full = encode_genres(small_full)

print(f"Total unique genres: {len(mlb.classes_)}")

In [ ]:

# Pretrained ResNet and Fine-Tuning
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load a pre-trained ResNet50
resnet_model = models.resnet50(pretrained=True)

# Modify the final classification head
num_ftrs = resnet_model.fc.in_features
num_genres = len(mlb.classes_)

# Replace the last layer for the 20 categories with a Sequential block containing Dropout
resnet_model.fc = nn.Sequential(
    nn.Dropout(0.5),               # Drops 50% of neurons for regularization
    nn.Linear(num_ftrs, num_genres)
)

# Freeze early layers: only train layer4 and the classification head (fc)
for name, param in resnet_model.named_parameters():
    if "layer4" not in name and "fc" not in name:
        param.requires_grad = False

#  Transfer the model to the GPU
resnet_model = resnet_model.to(device)

# Optimizer, Scheduler & Loss
# Using a lower learning rate and weight decay for fine-tuning
optimizer = torch.optim.Adam(resnet_model.parameters(), lr=0.00005, weight_decay=1e-4)

# Reduces LR by a factor of 10 every 2 epochs
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.1)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor.to(device))

epochs_resnet = 10
train_losses_res = []
val_losses_res = []

# EARLY STOPPING VARIABLES
best_val_loss = float('inf')
patience = 3
patience_counter = 0

print(f"Fine-tuning ResNet50 with Scheduler and Early Stopping on {device}...")

# TRAINING LOOP
for epoch in range(epochs_resnet):
    resnet_model.train()
    running_loss = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs_resnet} [Training]")

    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device).float()

        optimizer.zero_grad()
        outputs = resnet_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        progress_bar.set_postfix({'loss': loss.item(), 'lr': scheduler.get_last_lr()[0]})

    epoch_train_loss = running_loss / len(train_dataset)
    train_losses_res.append(epoch_train_loss)

    # metrics for loss curves
    resnet_model.eval()
    val_running_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device).float()
            outputs = resnet_model(images)
            loss = criterion(outputs, labels)
            val_running_loss += loss.item() * images.size(0)

    epoch_val_loss = val_running_loss / len(val_dataset)
    val_losses_res.append(epoch_val_loss)

    print(f"Epoch {epoch+1} Results: Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")

    #EARLY STOPPING CHECK
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        # Save the state of the model with the lowest validation loss
        torch.save(resnet_model.state_dict(), 'best_resnet_model.pth')
        patience_counter = 0
        print(f"--> Best model saved (Val Loss improved).")
    else:
        patience_counter += 1
        print(f"--> No improvement for {patience_counter} epoch(s).")

    if patience_counter >= patience:
        print(f"Early stopping triggered! Training stopped at epoch {epoch+1}")
        break

    # Update Learning Rate at the end of each epoch
    scheduler.step()


# VISUALIZE LOSS CURVES

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(train_losses_res) + 1), train_losses_res, label='Training Loss', marker='o', color='blue')
plt.plot(range(1, len(val_losses_res) + 1), val_losses_res, label='Validation Loss', marker='s', color='orange')
plt.title('ResNet50 Training vs Validation Loss', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

#  Quantitative Evaluation
# Load the best performing model saved during training before final evaluation
resnet_model.load_state_dict(torch.load('best_resnet_model.pth'))
resnet_model.eval()

all_targets_res = []
all_predictions_res = []

print("Evaluating the BEST model on the Test Set (ML-Small)...")

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = resnet_model(images)

        # Apply Sigmoid to get probabilities
        probs = torch.sigmoid(outputs)

        all_targets_res.append(labels.cpu().numpy())
        all_predictions_res.append(probs.cpu().numpy())

# Concatenate all batches
all_targets_res = np.vstack(all_targets_res)
all_predictions_res = np.vstack(all_predictions_res)

# Binary predictions based on a 0.3 threshold
binary_predictions_res = (all_predictions_res > 0.3).astype(int)

print(f"Evaluation Complete!")

# Calculate Metrics
macro_f1_res = f1_score(all_targets_res, binary_predictions_res, average='macro', zero_division=0)
micro_f1_res = f1_score(all_targets_res, binary_predictions_res, average='micro', zero_division=0)
roc_auc_res = roc_auc_score(all_targets_res, all_predictions_res, average='macro')

#  VISUALIZE QUANTITATIVE METRICS

metrics_dict_res = {'Macro F1': macro_f1_res, 'Micro F1': micro_f1_res, 'ROC-AUC': roc_auc_res}

plt.figure(figsize=(8, 5))
bars = plt.bar(metrics_dict_res.keys(), metrics_dict_res.values(), color=['#87CEEB', '#98FB98', '#FA8072'])
plt.ylim(0, 1.1)
plt.title('Test Set Evaluation Metrics (ResNet50 - Best Model)', fontsize=14)
plt.ylabel('Score', fontsize=12)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.02, f'{yval:.4f}',
             ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

Common questions:

1.  Which model performed better, and by how much?

2. Based on  the training and validation loss curves for both models, which one  converged faster? Where there any signs of overfitting in either model?

3. Why did the Pre-trained ResNet achieves these results compared to the Custom CNN, considering both models were trained on the exact same movie poster dataset?

### Αnswers

1) The evaluation metrics between of the two models confirm anyones' original hypothesis, that the pre-trained ResNet50 is superior to the custom CNN architecture one. Specifically, there was  a dramatic increase at the Macro F1 score (from 0.1585 to 0.3103), which clearly indicates  that ResNet is  more effective at identifying less frequent genres within the dataset. Furthermore, the improvement in ROC-AUC from 0.6673 to 0.7419 confirms that the ResNet model possesses a superior ability to distinguish between different classes, and as a results  reduces classification errors. This performance gap is not only  quantitative but qualitative as well, as the pre-trained model extracts more meaningful and valuable information from the visual data of the movie posters.




2)  The differences in the learning behavior of each model learns are clearly showcased in the analysis of their loss curves. The most obvious distinction is found in the convergence speed.  When it comes to the ResNet50, we can clearly see that this pre-trained model begins training with a wealth of visual knowledge already acquired from ImageNet, allowing it to stabilize its performance as early as the 4th epoch. On the other hand, the Custom CNN converges at much smoother and more gradualy, as it starts from scratch and must build every level of visual understanding, using only what is provided from the  dataset. Now as for the of overfitting problem, the two models present opposing behaviors. ResNet50 shows clear signs of overtraining, as we can clearly observe from the training-validation loss curves, as there is a significant gap between them. This may seem problematic but in reality it is completely normal, since the model's high computational capacity leads it to "memorize" the specific details of the training set. Also the use of the "Dropout" and "Early Stopping" stragetigies, successfully held the validation loss at a steady plateau, preventing a total collapse in generalization. On the other hand, the Custom CNN demonstrates a relatively  balanced training process with very little overfitting (especially in the first 25-30 epochs). This obviously suggests that the CNN  learns only features that it can effectively apply to unseen data. Last but not least, taking into consideration the 1st Question, we can conclude that while the Custom CNN is more stable during training, the ability of ResNet50 to understand complex visual patterns makes it the ideal choice for extracting the  visual embeddings required.






3)  Despite both models being trained on the exact same dataset, the main reason why the ResNet50 model outperforms the CNN, is because of its architecture and the prior knowledge it incorporates. Having been pre-trained on ImageNet, ResNet already recognizes fundamental visual elements such as edges, colours, shapes,  which it then fine-tunes for movie posters. Additionally, the use of Residual Connections  allows the network to maintain a depth of 50 layers without losing gradient information during training. Moreover no matter how well assembled,  the simple CNN was built from scratch, and just cannot capture complex patterns in an image. On the other hand ResNet50 has very little limitations. It can easily combine multiple features which is limited to capturing basic patterns, ResNet can synthesize complex features—such as (lighting, facial composition)that are crucial for accurately identifying a film's genre.

---
### **Extract Visual Embeddings**


Now that you have trained both a Custom CNN and a Pre-trained ResNet to understand movie genres based on visual features, we want to extract that "visual understanding" to use in our recommendation system graph (Task 3).

**Look back at your evaluation metrics (F1 Score, ROC-AUC) and choose the model that achieved the BEST overall results.**

Instead of getting the final genre predictions (the raw logits), we will pass the posters from the  `ml-latest-small` dataset through your chosen model and stop at the **penultimate layer**. This gives us a dense feature vector (embedding) for every single movie. (Note: The size of this vector will depend on the model you chose—for example, a standard ResNet18 outputs a 512-dimensional vector).

Ensure the order of the embeddings perfectly matches the dataframe! Save the resulting tensor to a `.pt` (PyTorch) file so it can be easily loaded in Task 3.


In [ ]:

# Choosing the best model(ResNet50)
best_vision_model = resnet_model

# Store the original fc layer
original_fc = best_vision_model.fc

# Replace the final classification layer with Identity(This ensures the model returns the output of the penultimate layer (2048 features))
best_vision_model.fc = nn.Identity()

# Setup DataLoader (shuffle MUST be False to maintain the correct order for movieIds!)
inference_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

best_vision_model.to(device)
best_vision_model.eval()
all_embeddings = []

print(f"Extracting embeddings for {len(test_dataset)} movies...")


# extraction loop

with torch.no_grad(): # Disable gradient calculation for inference efficiency
    for images, _ in tqdm(inference_loader, desc="Extracting"):
        # Move images to GPU
        images = images.to(device)

        # Forward pass: Get the feature embeddings
        embeddings = best_vision_model(images)

        # Move embeddings back to CPU and add to list
        all_embeddings.append(embeddings.cpu())


# Concatenate all batches into a single large tensor
final_visual_embeddings = torch.cat(all_embeddings, dim=0)
print(f"Extracted Embeddings Shape: {final_visual_embeddings.shape} -> [Num_Movies, Embedding_Dim]")

# Restore the original fc layer
best_vision_model.fc = original_fc


save_dict = {
    'movie_ids': torch.tensor(small_full['movieId'].values),
    'embeddings': final_visual_embeddings
}

# Create directory
if not os.path.exists('datasets'):
    os.makedirs('datasets')

# Save the extracted embeddings to a file
torch.save(save_dict, 'datasets/cv_embeddings.pt')
print("Visual embeddings successfully saved to 'datasets/cv_embeddings.pt'!")

---
##Natural Language Processing for Genre Classification

Now we will tackle the exact same multi-label classification problem, but using a completely different modality: **Text**. We will build a sequence model (RNN, LSTM, or GRU) to predict genres based on the movie's plot summary (the `overview` column). To make this model more powerful and interpretable, a **Custom Attention Mechanism** will be implemented

### **Text Preprocessing & Data Loading**
Unlike images, neural networks cannot process raw text. We must convert the words into numerical tokens. Below, we provide the pipeline to clean and tokenize the text, build a Vocabulary based solely on the training set (to avoid data leakage) and convert summaries into padded numerical tensors.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Assuming datasets.zip is in the root of your Google Drive
!unzip -o /content/drive/MyDrive/datasets.zip -d datasets/

In [ ]:
import pickle as pkl
import pandas as pd


# load pre-saved pickled dataframes, if needed
train_df = pkl.load(open('train_df.pkl', 'rb'))
val_df = pkl.load(open('val_df.pkl', 'rb'))

A tokenizer is essentially the "translator" that turns a raw blob of text into a format a machine can actually digest.

Complete the simple tokenizer, as instructed below.

In [ ]:
# Simple Tokenizer
def tokenize(text):
    # input text to a lowercase string,remove any character that isn't a letter, number, or whitespace, and split the cleaned string into a list of tokens.
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    tokens = text.split()
    return tokens

# Build Vocabulary from Training Set ONLY
print("Building vocabulary...")
all_tokens = []
for plot in train_df['overview']:
    all_tokens.extend(tokenize(plot))

We need to build a mapping of the words within the plot summaries. We select our vocabulary to contain the top 10,000 most frequent words and map them to unique IDs. It reserves 0 for padding and 1 for unknown words (the rare ones we excluded). Raw text needs to be converted into fixed-length numerical vectors (here, set to 150 tokens), truncating long sequences and padding short ones to ensure uniform input for the model.

In [ ]:
# Keep top 10,000 most common words, reserve 0 for <PAD> and 1 for <UNK>

max_vocab_size = 10000
vocab_counts = Counter(all_tokens)
vocab = {word: idx + 2 for idx, (word, _) in enumerate(vocab_counts.most_common(max_vocab_size))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1

def text_to_tensor(text, vocab, max_len=150):
    """Converts a text string to a padded tensor of token IDs."""
    tokens = tokenize(text)
    token_ids = [vocab.get(word, vocab['<UNK>']) for word in tokens]

    # Truncate if too long
    if len(token_ids) > max_len:
        token_ids = token_ids[:max_len]

    # Pad if too short
    pad_len = max_len - len(token_ids)
    token_ids = token_ids + [vocab['<PAD>']] * pad_len

    return torch.tensor(token_ids, dtype=torch.long)

And we will build a pytorch dataset like before.

In [ ]:
class MoviePlotDataset(Dataset):
    def __init__(self, dataframe, vocab):
        self.dataframe = dataframe.reset_index(drop=True)
        self.vocab = vocab
        self.label_cols = self.dataframe.columns[6:].tolist()

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        plot_text = self.dataframe.loc[idx, 'overview']
        text_tensor = text_to_tensor(plot_text, self.vocab)

        # Extract labels
        labels = self.dataframe.loc[idx, self.label_cols].values.astype(float)
        labels = torch.tensor(labels, dtype=torch.float32)

        return text_tensor, labels

In [ ]:
print("Initializing NLP Datasets...")
nlp_train_dataset = MoviePlotDataset(train_df, vocab)
nlp_val_dataset = MoviePlotDataset(val_df, vocab)
nlp_test_dataset = MoviePlotDataset(small_full, vocab)

In [ ]:
batch_size = 32
nlp_train_loader = DataLoader(nlp_train_dataset, batch_size=batch_size, shuffle=True)
nlp_val_loader = DataLoader(nlp_val_dataset, batch_size=batch_size, shuffle=False)
nlp_test_loader = DataLoader(nlp_test_dataset, batch_size=batch_size, shuffle=False)

# Test the dataloader
print("For training...")
sample_texts, sample_labels = next(iter(nlp_train_loader))
print(f"Text Batch Shape: {sample_texts.shape} -> [Batch, Sequence_Length]")
print(f"Label Batch Shape: {sample_labels.shape} -> [Batch, Num_Genres]")

---
### **Architecture with Custom Attention**

Now we build a deep learning text classifier. However, instead of simply passing the final hidden state to the classification head, we implement aSelf-Attention Mechanism to compute a weighted context vector from *all* the hidden states of the sequence.

**A simple Attention Mechanism:**
Given the hidden states of the RNN layer:
1. We calculate the attention scores for each token: $e_t = \text{Linear}(h_t)$
2. Then we Compute the normalized attention weights using Softmax: $\alpha = \text{softmax}(e)$
3. And then we compute the final context vector by taking the weighted sum of the hidden states: $c = \sum_{t=1}^{T} \alpha_t h_t$


In [ ]:
class CustomTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super(CustomTextClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)# map vocab_size to embed_dim with padding_idx=0

        self.embedding_dropout = nn.Dropout(0.3)    #avoid overfitting

        self.rnn = nn.GRU(input_size=embed_dim, hidden_size=hidden_dim, batch_first=True, bidirectional=True) #batch_first=True

        self.context_dropout = nn.Dropout(0.3)

        self.attention_proj = nn.Linear(hidden_dim * 2, 1)

        self.fc = nn.Linear(hidden_dim * 2, num_classes) #a Linear layer mapping hidden_dim to num_classes


    def forward(self, x):
        #FORWARD PASS
        #Pass input through the embedding layer.
        #Pass embeddings through the RNN. Get all hidden states 'H'.
        #Calculate energy scores 'e'
        #Calculate attention weights 'alpha'.
        #Calculate the context vector 'c'.
        #Pass the context vector through the final classifier to get logits.

        x = self.embedding(x)
        x = self.embedding_dropout(x)
        H, _ = self.rnn(x)
        e = self.attention_proj(H).squeeze(-1)
        alpha = F.softmax(e, dim=1)
        c = torch.sum(H * alpha.unsqueeze(-1), dim=1)
        c = self.context_dropout(c)
        logits = self.fc(c)
        return logits



    def extract_embeddings(self, x):
        """Returns the final context vector 'c' BEFORE the classification head."""
        # We Replicate the forward pass logic above, but stop after calculating
        # Return the context vector 'c' as the embedding.
        x = self.embedding(x)
        x = self.embedding_dropout(x)
        H, _ = self.rnn(x)

        e = self.attention_proj(H).squeeze(-1)
        alpha = F.softmax(e, dim=1)

        c = torch.sum(H * alpha.unsqueeze(-1), dim=1)

        return c


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Model will be initialized in {device}.\n")

vocab_size = len(vocab)
embed_dim = 64
hidden_dim = 64  #smaller model -> solves the overfitting problem and has better generalization
num_classes = len(nlp_train_dataset.label_cols)

nlp_model = CustomTextClassifier(vocab_size, embed_dim, hidden_dim, num_classes).to(device)
print(nlp_model)

---
### **Training & Evaluation Loop**
Now we train and evaluate the model! Then plot the Training and Validation loss curves and finally evaluate the model on ML-Small summaries using Macro/Micro F1-Scores.

In [ ]:

#Initialize the loss criterion and optimizer (with pose_weight)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer = torch.optim.AdamW(nlp_model.parameters(), lr=5e-4, weight_decay=1e-4)   #regulariazation καλύτερα από Αdam σε περιπτώσεις overfitting

epochs = 20
train_losses = []
val_losses = []

#TRAINING & VALIDATION LOOP
for epoch in range(epochs):
    #TRAINING PHASE
    nlp_model.train()
    running_train_loss = 0.0

    #training loop
    #Iterate over train_loader and then Forward pass, Calculate Loss, Backward pass, Optimizer step (we will use gradient clipping to prevent expoding gradients)
    for texts, labels in nlp_train_loader:
        texts, labels = texts.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = nlp_model(texts)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(nlp_model.parameters(), max_norm=1)
        optimizer.step()

        running_train_loss += loss.item() * texts.size(0)

    train_loss = running_train_loss / len(nlp_train_loader)



    #  VALIDATION PHASE
    nlp_model.eval()
    running_val_loss = 0.0
    # Iterate over val_loade, Forward pass, Calculate Loss

    with torch.no_grad():
        for texts, labels in nlp_val_loader:
            texts, labels = texts.to(device), labels.to(device)

            outputs = nlp_model(texts)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * texts.size(0)

    val_loss = running_val_loss / len(nlp_val_loader)



    # Save average losses for plotting
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")



# VISUALIZATION LOSS CURVES
plt.figure(figsize=(10, 5))
plt.plot(range(1, epochs+1), train_losses, label='Training Loss', marker='o', color='blue')
plt.plot(range(1, epochs+1), val_losses, label='Validation Loss', marker='o', color='orange')
plt.title('NLP Model Loss Over Epochs', fontsize=14)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('Loss (Weighted BCE)', fontsize=12)
plt.xticks(range(1, epochs+1))
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

In [ ]:
torch.save(nlp_model.state_dict(), "/content/drive/MyDrive/nlp_model_best4.pth")

In [ ]:
#EVALUATION
from sklearn.metrics import f1_score, roc_auc_score

print("\nEvaluating on Test Set...")

nlp_model.eval()
all_targets = []
all_predictions = []

with torch.no_grad():
    for texts, labels in nlp_test_loader:
        texts, labels = texts.to(device), labels.to(device)

        logits = nlp_model(texts)
        probs = torch.sigmoid(logits)

        all_targets.append(labels.cpu().numpy())
        all_predictions.append(probs.cpu().numpy())

all_targets = np.vstack(all_targets)
all_predictions = np.vstack(all_predictions)
binary_predictions = (all_predictions >= 0.5).astype(int)


macro_f1 = f1_score(all_targets, binary_predictions, average='macro', zero_division=0)
micro_f1 = f1_score(all_targets, binary_predictions, average='micro', zero_division=0)
roc_auc = roc_auc_score(all_targets, all_predictions, average='macro')

print(f"Test Macro F1-Score: {macro_f1:.4f}")
print(f"Test Micro F1-Score: {micro_f1:.4f}")

# VISUALIZE the metrics
metrics_dict = {'Macro F1': macro_f1, 'Micro F1': micro_f1, 'ROC-AUC': roc_auc}

plt.figure(figsize=(8, 5))
bars = plt.bar(metrics_dict.keys(), metrics_dict.values(), color=['#87CEEB', '#98FB98', '#FA8072'])
plt.ylim(0, 1.1)
plt.title('Test Set Evaluation Metrics', fontsize=14)
plt.ylabel('Score', fontsize=12)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.02, f'{yval:.4f}',
             ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()
pos_weight_tensor

---
### **Extract NLP Embeddings for the GNN**
Now we need to extract the semantic "understanding" of the plots to use as node features in the GNN later on, and save them in `nlp_embeddings.pt`. Firstly, We create a list named `all_nlp_embeddings` to store batch results. Then extract the embeddings and move the resulting embeddings back to the CPU and append them to the list. Finally we combine all batches into a single tensor named `final_nlp_embeddings`.


In [ ]:
torch.load('/content/drive/MyDrive/nlp_model_best4.pth')
nlp_model.load_state_dict(torch.load('/content/drive/MyDrive/nlp_model_best4.pth'))

In [ ]:
# EXTRACTION LOOP
nlp_model.eval()
all_nlp_embeddings = []

with torch.no_grad():
    for texts, _ in nlp_test_loader:
        texts = texts.to(device)

        embeddings = nlp_model.extract_embeddings(texts)
        all_nlp_embeddings.append(embeddings.cpu())

final_nlp_embeddings = torch.cat(all_nlp_embeddings, dim=0)

# SAVING
print(f"Extracted NLP Embeddings Shape: {final_nlp_embeddings.shape} -> [Num_Movies, Hidden_Dim]")

save_dict = {
    'movie_ids': torch.tensor(small_full['movieId'].values),
    'embeddings': final_nlp_embeddings
}

torch.save(save_dict, 'datasets/nlp_embeddings.pt')
print("Semantic NLP embeddings successfully saved to 'datasets/nlp_embeddings.pt'!")

---
### **Common Questions**

1. Did the NLP model (Plot Summaries) or the CV model (Movie Posters) perform better at predicting genres, and why?
2. We capped the `max_len` at 150 words and padded shorter sequences with 0s. How might processing very long padding sequences negatively affect an RNN's performance, and how does the Attention mechanism help mitigate this issue?

1. The NLP model performed better than the CV model in genre prediction. This is clear from the final evaluation metrics, since the NLP model achieved higher Macro F1, Micro F1, and ROC-AUC scores than the CNN model. Specifically, the NLP model reached 0.2798 Macro F1, 0.4719 Micro F1, and 0.7483 ROC-AUC, while the CNN model achieved 0.1585 Macro F1, 0.3824 Micro F1, and 0.6673 ROC-AUC. We believe the NLP modality was more successful because the plot summaries contain direct semantic information about the story, themes, setting, and events of a movie. Genres such as Western, Sci-Fi, Crime, War, or Mystery are often explicitly reflected in words and phrases inside the overview. In contrast, movie posters provide only indirect visual cues, which are often ambiguous. Many genres may share similar poster designs, making it harder for a CNN to distinguish them reliably.


2. We capped the maximum sequence length at 150 words and padded shorter sequences with zeros so that all inputs would have the same shape. Very long padded sequences can negatively affect an RNN because they force it to process many irrelevant positions which introduce noise and thus making learning less efficient. In addition, RNNs may struggle to preserve important information over long sequences. The Attention mechanism helps mitigate this issue because it looks at all hidden states and assigns higher weights to the most important words or positions in the sequence. As a result, even if the summary is long, the model can still focus on the most important parts of the text when forming the final context vector.

---
##Graph Neural Networks for Link Prediction**


Now we have reached the point we we will build the recommendation system. We will construct a **Bipartite Heterogeneous Graph** consisting of `User` nodes and `Movie` nodes, connected by `rates` edges.The goal is **Link Prediction**, where we  predict the probability that a specific User will rate a specific Movie. We will  engineer the node features using the visual and textual embeddings that where extracted before, format the graph data, and perform an ablation study to explain why multimodal features improve recommendation systems.



---
### **Graph Construction**
Graph frameworks require node IDs to be contiguous integers starting from $0$. MovieLens IDs (like `movieId = 5000`) will cause out-of-bounds errors if not mapped properly. The code below loads the ratings and creates mappings.

In [ ]:
!unzip -o /content/drive/MyDrive/datasets.zip -d datasets/

In [ ]:
print("Installing PyTorch Geometric base...")
!pip install -q torch-geometric

print("Attempting to install optional C++ speedups (this may take a minute)...")
torch_version = torch.__version__.split('+')[0]
cuda_version = torch.version.cuda.replace('.', '')
whl_url = f"https://data.pyg.org/whl/torch-{torch_version}+cu{cuda_version}.html"

os.system(f"pip install -q pyg_lib torch_scatter torch_sparse -f {whl_url} > /dev/null 2>&1")

print("PyTorch Geometric environment ready!")

In [ ]:
# Load the ratings data from the small graph
ratings_df = pd.read_csv('datasets/datasets/ml-latest-small/ratings.csv')

# ratings for movies that exist in  valid small_full set
small_full = pd.read_csv('datasets/datasets/ml_small_metadata_full.csv')
valid_movie_ids = set(small_full['movieId'])
ratings_df = ratings_df[ratings_df['movieId'].isin(valid_movie_ids)]

# Create Contiguous IDs for Users and Movies
unique_users = ratings_df['userId'].unique()
unique_movies = small_full['movieId'].unique() # Use the full valid subset, not just rated ones

user_mapping = {userid: i for i, userid in enumerate(unique_users)}
movie_mapping = {movieid: i for i, movieid in enumerate(unique_movies)}

# Map the edges
src = [user_mapping[uid] for uid in ratings_df['userId']]
dst = [movie_mapping[mid] for mid in ratings_df['movieId']]
edge_index = torch.tensor([src, dst], dtype=torch.long)

print(f"Total Users: {len(user_mapping)}")
print(f"Total Movies: {len(movie_mapping)}")
print(f"Total Ratings (Edges): {edge_index.shape[1]}")


cv_data = torch.load('datasets/cv_embeddings.pt')
cv_features = cv_data['embeddings'] # Shape: [Num_Movies, 512]

nlp_data = torch.load('datasets/nlp_embeddings.pt')
nlp_features = nlp_data['embeddings'] # Shape: [Num_Movies, 256]

---
### **Feature Engineering & Initialization**

A GNN relies on its starting node features. Because we don't have metadata for the users (like age or gender), we will initialize user nodes with random learnable embeddings. For the movies, you will test different feature combinations.


In [ ]:
cv_data = torch.load('datasets/cv_embeddings.pt')
cv_features = cv_data['embeddings'] # Shape: [Num_Movies, 512]


nlp_data = torch.load('datasets/nlp_embeddings.pt')
nlp_features = nlp_data['embeddings'] # Shape: [Num_Movies, 256]

We ensure that the order of the embeddings matches the contiguous `movie_mapping`. They should already match since we used `small_full` sequentially. It is very important to normalize the extracted features! CV and NLP embeddings are raw outputs from custom networks and their range will blow up the gradients.

In [ ]:
def get_movie_features(strategy, cv_features, nlp_features, num_movies):
    """Returns the appropriate node feature tensor based on the selected ablation strategy."""
    if strategy == "baseline":
        return torch.randn((num_movies, 64))

    elif strategy == "cv":
      return F.normalize(cv_features, p=2, dim=1)
         # Return the L2-normalized visual embeddings

    elif strategy == "nlp":
      return F.normalize(nlp_features, p=2, dim=1) #p=2 L2 and dim=1 for row
         #Return the L2-normalized visual embeddings

    elif strategy == "multimodal":
       # Concatenate the CV and NLP features,then apply L2-normalization to the combined vector.
        cv_norm = F.normalize(cv_features, p=2, dim=1)
        nlp_norm = F.normalize(nlp_features, p=2, dim=1)
        multimodal_feats = torch.cat([cv_norm, nlp_norm], dim=1)
        return F.normalize(multimodal_feats, p=2, dim=1)


    elif strategy == "pretrained":
        pretrained_data = torch.load('pretrained_nlp_embeddings.pt')
        #dictionary,key 'embeddings'
        p_feats = pretrained_data['embeddings'] if isinstance(pretrained_data, dict) else pretrained_data
        return F.normalize(p_feats, p=2, dim=1)
         # Load the 'pretrained_nlp_embeddings.pt' file and return the normalized embeddings.

    else:
        raise ValueError("Invalid strategy! Please choose 'baseline', 'cv', 'nlp', 'multimodal', or 'pretrained'.")

This function transforms raw data into a **Bipartite Heterogeneous Graph**. In high-level terms, it sets up the "map" that the GNN will use to navigate and learn.

* **`HeteroData()`**: Instead of a simple list, we initialize a specialized container that recognizes different "species" of nodes (Users vs. Movies) and their specific relationships.
* **User Nodes (Structural IDs)**: We tell the graph how many users exist. Since we have no metadata for them, the GNN will later assign them unique, learnable "fingerprints" (embeddings) based purely on their position in the graph.
* **Movie Nodes (Semantic Features)**: Unlike users, movies are initialized with the rich features you extracted (CV/NLP). This allows the GNN to understand *what* a movie is, rather than just knowing its ID.
* **Edge Index (The Connections)**: This maps the "rates" relationship. It acts as the bridge that tells the model which users interacted with which movies.
* **`T.ToUndirected()` (Two-Way Communication)**: By default, a rating is one-way (User $\to$ Movie). This transform creates "reverse" edges so that information can flow from movies back to users, allowing the GNN to perform complex message passing.

In [ ]:
def build_graph(movie_features):
    data = HeteroData()

    # User nodes just need to know how many exist
    data['user'].num_nodes = len(user_mapping)

    # Movie nodes require the actual semantic features
    data['movie'].x = movie_features

    # Add Edges
    data['user', 'rates', 'movie'].edge_index = edge_index

    # GNNs need undirected paths to pass messages back and forth
    data = T.ToUndirected()(data)

    return data

---
### **GNN Architecture and Training**

Specifically:
* **The Encoder:** Two layers of message passing. It aggregates features from neighboring nodes to create rich, relational embeddings for both users and movies.
* **The Decoder (Link Predictor):** To predict if an edge exists, we take the final GNN embedding of a User and a Movie, and compute their dot product. We then pass this through a Sigmoid to get a probability.

In [ ]:
# add_self_loops=False and edge features  #project=True
class HeteroGNNEncoder(nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        # HeteroConv allows us to define specific message passing for specific edge types.
        self.conv1 = HeteroConv({
            ('user', 'rates', 'movie'): SAGEConv((-1, -1), hidden_channels, root_weight=True),
            ('movie', 'rev_rates', 'user'): SAGEConv((-1, -1), hidden_channels, root_weight=True),
        }, aggr='mean')

        self.conv2 = HeteroConv({
            ('user', 'rates', 'movie'): SAGEConv((-1, -1), out_channels, root_weight=True),
            ('movie', 'rev_rates', 'user'): SAGEConv((-1, -1), out_channels,root_weight=True),
        }, aggr='mean')

    def forward(self, x_dict, edge_index_dict):
        x_initial = {key: x for key, x in x_dict.items()}
        x_dict = self.conv1(x_dict, edge_index_dict)
        x_dict = {key: F.relu(x) for key, x in x_dict.items()}
        x_dict = {key: F.dropout(x, p=0.3, training=self.training) for key, x in x_dict.items()} #το30%
        x_dict = self.conv2(x_dict, edge_index_dict)
        x_dict = {key: x + x_initial[key] for key, x in x_dict.items()} #skip connection
        return x_dict


class EdgeDecoder(nn.Module):
    def __init__(self, hidden_channels):
        super().__init__()
        # Decodes the probability of a link by concatenating User and Movie embeddings
        self.lin1 = nn.Linear(2 * hidden_channels, hidden_channels)
        self.lin2 = nn.Linear(hidden_channels, 1)

    def forward(self, z_dict, edge_label_index):
        row, col = edge_label_index
        # Get the embeddings for the source (User) and destination (Movie)
        z = torch.cat([z_dict['user'][row], z_dict['movie'][col]], dim=-1)
        z = F.relu(self.lin1(z))
        z = self.lin2(z)
        return z.view(-1)


class RecommenderGNN(nn.Module):
    def __init__(self, hidden_channels, num_users, movie_feature_dim):
        super().__init__()
        # 1. Structural Embedding for Users (They have no metadata)
        self.user_emb = nn.Embedding(num_users, hidden_channels)

        # 2. Projection Layer for Movies
        self.movie_proj = nn.Linear(movie_feature_dim, hidden_channels)

        self.encoder = HeteroGNNEncoder(hidden_channels, hidden_channels)
        self.decoder = EdgeDecoder(hidden_channels)

    def forward(self, x_dict, edge_index_dict, edge_label_index):
        # Initialize node representations
        h_dict = {
            'user': self.user_emb.weight,
            'movie': self.movie_proj(x_dict['movie'])
        }

        # Pass through GraphSAGE layers
        z_dict = self.encoder(h_dict, edge_index_dict)

        # Decode the requested edges
        return self.decoder(z_dict, edge_label_index)

The CNN and RNN were trained on one set of data and evaluated on completely unseen, isolated test data. However Graph machine learning often relies on transductive learning. The model sees the entire graph structure (all User and Movie nodes) during the training phase. Because we can't easily isolate nodes without breaking the graph, we split **edges** instead of nodes. We hide a percentage of the "rates" edges (10% of the rating edges for validation and 10% for final testin) to serve as our test set. The model learns by passing messages across the visible training edges, and we evaluate it by asking it to predict the existence of the edges we hid.


In [ ]:
#TRAINING & EVALUATION LOOP
def train_and_evaluate_graph(graph_data, num_epochs=50, hidden_channels=64, lr=0.01):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Higher Negative Sampling for accuracy
    transform = T.RandomLinkSplit(
        num_val=0.1, num_test=0.1, disjoint_train_ratio=0.3,
        neg_sampling_ratio=2.0,
        add_negative_train_samples=True,
        edge_types=[('user', 'rates', 'movie')],
        rev_edge_types=[('movie', 'rev_rates', 'user')]
    )

    train_data, val_data, test_data = transform(graph_data)
    train_data, val_data = train_data.to(device), val_data.to(device)

    model = RecommenderGNN(hidden_channels, graph_data['user'].num_nodes, graph_data['movie'].x.shape[1]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    # 2. Scheduler: Reduces LR if the AUC does not improve
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

    criterion = torch.nn.BCEWithLogitsLoss()
    history = {'loss': [], 'val_auc': []}

    best_val_auc = 0
    patience_counter = 0

    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()
        out = model(train_data.x_dict, train_data.edge_index_dict, train_data['user', 'rates', 'movie'].edge_label_index)
        loss = criterion(out, train_data['user', 'rates', 'movie'].edge_label.float())
        loss.backward()
        optimizer.step()

        # Validation
        model.eval()
        with torch.no_grad():
            val_out = model(val_data.x_dict, val_data.edge_index_dict, val_data['user', 'rates', 'movie'].edge_label_index)
            val_auc = roc_auc_score(val_data['user', 'rates', 'movie'].edge_label.cpu(), torch.sigmoid(val_out).cpu())

        history['loss'].append(loss.item())
        history['val_auc'].append(val_auc)

        #  Scheduler update
        scheduler.step(val_auc)

        # 3. Early Stopping
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= 7:
            print(f"Early stopping at epoch {epoch}")
            break

    return history, model



In [ ]:
num_movies = len(movie_mapping)
strategies = ['baseline', 'cv', 'nlp', 'multimodal', 'pretrained']

In [ ]:
num_movies = len(movie_mapping)
strategies = ['baseline', 'cv', 'nlp', 'multimodal', 'pretrained']
results = {}
for strategy in strategies:
    print(f"Training Strategy: {strategy.upper()}")

    #features using the helper function
    features = get_movie_features(strategy, cv_features, nlp_features, num_movies)

    # Build the PyG HeteroData Graph
    graph = build_graph(features)

    #Train and Evaluate
    history, _ = train_and_evaluate_graph(graph, num_epochs=40)
    results[strategy] = history

# PLOTTING THE COMPARISON
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
styles = ['--', '-.', ':', '-', '-']
colors = ['gray', 'blue', 'green', 'purple', 'red']

for i, (name, history) in enumerate(results.items()):
    epochs = range(1, len(history['loss']) + 1)

    ax1.plot(epochs, history['loss'], linestyle=styles[i], color=colors[i], label=name.upper(), linewidth=2)
    ax2.plot(epochs, history['val_auc'], linestyle=styles[i], color=colors[i], label=name.upper(), linewidth=2)

ax1.set_title('Training Loss (BCE) over Epochs', fontsize=14)
ax1.set_xlabel('Epochs', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.set_title('Validation ROC-AUC over Epochs', fontsize=14)
ax2.set_xlabel('Epochs', fontsize=12)
ax2.set_ylabel('ROC-AUC Score', fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

---
### **Common Questions**

1. If the movie features are random noise, how is the model still successfully predicting links? What does this tell you about the power of the Graph Structure (the edges) versus the Node Features?

2. Is there significant differences between using one modality over the other?

3. Did the pretrained embeddings result in better performance?

Answers:
1.) The model can predict links successfully even with random noise because it relies on the Graph Structure. The graph is bipartite, meaning the edges represent the real-life interactions between the two kinds of nodes: users and movies.The GraphSAGE algorithm we used works through  "Message Passing". In every training layer, user nodes basically interact with the movie nodes, and they exchange information. Even if we start with random noise, after a few layers, the model creates embeddings ( vectors) that indicate the topological position of each node. Essentially the model basically  realizes that if two users have watched the same movies, which means that these two user nodes connect with the same movie node, then they are structurally similar. Therefore, it predicts that they will probably like other movies from their common neighborhood. This neighbourhood simply does not take into consideration the visual or the text embeddings of a movie. This proves that the Graph Structure is the most important signal in recommendation systems.  The edges of the graph (user behavior) contains much more important information for predicting future links than individual movie features do. However this does not mean that these Node Features are worthless. There are numerous cases in which they are necessary. If a new movie appears in the graph for example, that nobody has seen (no edges/ratings)  then the movie features appear to be the only way to provide a successful recommendation.

2.) Based on the Validation ROC-AUC plot, there are indeed significant differences between the modalities, especially in how fast the model learns and the final accuracy it achieves. The most noticeable difference is between NLP and CV. The NLP modality  is clearly the strongest. It rises much faster than any other line in the first 10 epochs and maintains a lead for most of the training. This shows that the text provides  much more powerful signal for the model to predict user-movie links compared to the movie posters.On the other hand, the CV modality  is consistently the weakest performer. Throughout the training, the CV curve stays below the others, which proves that using only visual information from posters is not as effective for recommendation tasks as using text. Now as for the Multimodal we can observe that  by the end of 40 epochs, it reaches the highest performance suggests that combining both text and images provides the most complete information for the system as expected.

3.)Based on the plots, the Pretrained embeddings resulted in very high performance, but they did not significantly outperform the Multimodal strategy. This probably happened because the model reaches a point where the performance simply does not improve. The Graph Structure is already so strong that any extra information from pretrained features only offers very little improvement. As mentioned in the 1st question, the Message Passing mechanism allows the model to learn powerful representations based on the topological position of the nodes. Thus, the model captures most of the necessary information through the graph's connectivity and not from the pre-trained embeddings. Additionally, pretrained models are often trained on general objects, while movie posters are really artistic and colorful, so they do not offer a massive advantage over the features we extracted from Tasks 1 and 2. In conclusion, while the pretrained embeddings are very stable and reliable, they didn't result in a significant boost.

---
### **Qualitative analysis**

While AUC scores and loss curves are essential for measuring model performance, the true test of a recommendation system is its ability to provide tangible, human-readable results.

In this section, we ask the model to predict the existence of *entirely new* edges, and print the results. We want to generate a "Top-5" list for a specific user. In order to do that we must first identify the movies this specific user has already rated in the graph. We do this to ensure we don't recommend movies they have already seen. Then we identify every movie in the `movie_mapping` that does not appear in the user's history. After that we construct a set of "potential edges" between the User and every possible candidate movie. We pass these into the trained GNN to calculate a Probability (0 to 1). After that comes the sorting of the candidate movies by their probability scores and then we select the top $K$.


In [ ]:
def recommend_movies_for_user(user_id, model, graph_data, metadata_df, user_mapping, movie_mapping, top_k=5):
    """Generates human-readable movie recommendations for a specific user in the following fomat:
    --- USER X HISTORY ---
    They have already rated N movies. Here are a few:
        Movie 1
        Movie 2
        Movie 3
        Movie 4
        Movie 5


    --- TOP 5 GNN RECOMMENDATIONS ---
        Movie 1 (Probability)
        Movie 2 (Probability)
        Movie 3 (Probability)
        Movie 4 (Probability)
        Movie 5 (Probability)
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    graph_data = graph_data.to(device)
    model.eval()

    # Reverse the `movie_mapping` dictionary to convert Graph IDs back to MovieLens IDs.
    inv_movie_mapping = {v: k for k, v in movie_mapping.items()}
    movie_titles = dict(zip(metadata_df['movieId'], metadata_df['title']))

    # We Find the internal Graph ID for the requested `user_id` using `user_mapping`.
    if user_id not in user_mapping:
        print(f"User {user_id} not found in the graph.")
        return

    # the real graph_id
    u_graph_id = user_mapping[user_id]

    #all the movies the user has rated.
    edge_index = graph_data['user', 'rates', 'movie'].edge_index
    mask = edge_index[0] == u_graph_id
    rated_movie_indices = edge_index[1, mask].cpu().numpy()

    print(f"--- USER {user_id} HISTORY ---")
    print(f"They have already rated {len(rated_movie_indices)} movies. Here are a few:")
    for m_idx in rated_movie_indices[:5]:
        m_id = inv_movie_mapping[m_idx]
        print(f"   {movie_titles.get(m_id, 'Unknown Title')}")

    # all the candidate movies (Movies that the  user has NOT rated yet).
    num_movies = graph_data['movie'].num_nodes
    all_movie_indices = np.arange(num_movies)
    candidate_indices = np.setdiff1d(all_movie_indices, rated_movie_indices)

    # `pred_edge_index` tensor of shape [2, num_candidates].
    u_idx_repeated = torch.full((len(candidate_indices),), u_graph_id, dtype=torch.long)
    m_idx_candidates = torch.from_numpy(candidate_indices).long()
    pred_edge_index = torch.stack([u_idx_repeated, m_idx_candidates], dim=0).to(device)

    # Forward Pass: the GNN  predicts the probability of these candidate edges.
    with torch.no_grad():
        preds = model(graph_data.x_dict,
                      graph_data.edge_index_dict,
                      pred_edge_index)
        probs = torch.sigmoid(preds).cpu().numpy()

    # 7. Sort the predictions
    top_indices = probs.argsort()[-top_k:][::-1]

    print(f"\n--- TOP {top_k} GNN RECOMMENDATIONS ---")
    for i in top_indices:
        m_idx = candidate_indices[i]
        m_id = inv_movie_mapping[m_idx]
        print(f"   {movie_titles.get(m_id, 'Unknown Title')} ({probs[i]:.4f})")

In [ ]:
graph = build_graph(get_movie_features("cv", cv_features, nlp_features, num_movies))
_, model = train_and_evaluate_graph(graph, num_epochs=40)

recommend_movies_for_user(
    user_id=42,
    model=model, #the trained GNN
    graph_data=graph, # Pass the full graph object so it has the structural context
    metadata_df=small_full, # The dataframe with movie titles
    user_mapping=user_mapping,
    movie_mapping=movie_mapping,
    top_k=5
)